# MoXpert — เครือข่าย Router (Router Network - สมการใน Paper ข้อ 4-5)

`p = sigmoid(MLP(V_fuse))`

**Router Network** คือเกต (Gate) ที่ทำหน้าที่ตัดสินใจสำหรับภาพคำถามและข้อความคำถาม (query image, query text) ว่า **ควรจะเปิดใช้งานผู้เชี่ยวชาญ (Experts) คนไหนบ้าง** ซึ่งต่างจาก Sparse-LLM MoE ทั่วไป (ที่เป็น Homogeneous FFN Experts + Softmax บน Simplex) เนื่องจาก MoXpert ใช้ **4 Heterogeneous Functional Experts** และ Router จะปล่อย **ความน่าจะเป็นในการเปิดใช้งานแบบอิสระแยกตามผู้เชี่ยวชาญแต่ละคน** ผ่าน **Multi-label Sigmoid (ไม่ใช่ Softmax)**

| สมการใน Paper | คำอธิบายสมการ | ตำแหน่งใน Notebook นี้ |
|-------|----------|------------------------|
| Eq. 1-3 | `V_img`, `V_text`, `V_fuse = [V_img ; V_text]` | *ส่วนเสริม (Optional)* เซลล์ CLIP Encoder |
| **Eq. 4** | **`p = sigmoid(MLP(V_fuse))`, `p ∈ [0,1]^N`** | คลาส `RouterMLP` |
| **Eq. 5** | การเทรน Router ด้วย BCE Loss | เซลล์สำหรับการเทรน (Training cell) |

- `V_fuse` มีขนาด **1152 มิติ (1152-d)** = `2 × 576` (โดยแต่ละ Modality มีขนาด `d = 576`)
- `N = 4` ผู้เชี่ยวชาญ (Experts): **Reference Extractor, Knowledge Guide, Reasoning Expert, Decision Maker**
- การตัดสินใจเปิดใช้งานเชิงทวิภาค (Binary activation) จะใช้ **ค่าเกณฑ์การตัดสินใจแยกตามผู้เชี่ยวชาญ (Per-expert threshold)** `τ_i` (ค่าเริ่มต้นคือ `0.5`): `y_i = 1 ถ้า p_i > τ_i ไม่เช่นนั้นเท่ากับ 0` (strictly greater ตาม Algorithm 1)

**รองรับการทำงานทั้งบน** เครื่อง Local macOS (MPS/CPU) **และ** Google Colab / Cloud GPU (CUDA)
Notebook นี้ **สมบูรณ์ในตัวเอง (Self-contained)** — มีการนิยามโค้ดทุกอย่างไว้ภายใน และต้องการเพียง `torch`, `numpy`, `scikit-learn` สำหรับระบบหลักและการประมวลผล ส่วน CLIP Encoder ของจริง (Eq. 1-3) จะอยู่ใน **เซลล์เสริม (Optional)** ซึ่งจะข้ามการทำงานอัตโนมัติหากไม่ได้ติดตั้งไลบรารี `clip`

## 1. การตรวจจับสภาพแวดล้อมและติดตั้งไลบรารีที่จำเป็น (Environment detection & dependencies)

ตรวจสอบว่ากำลังทำงานบน Colab หรือ เครื่อง Local และติดตั้งเฉพาะไลบรารีที่ยังไม่มี เซลล์หลักถูกออกแบบให้ใช้ไลบรารีเบาที่สุดเพื่อให้ชุดทดสอบทำงานได้เสมอ

In [1]:
# --- การตรวจจับสภาพแวดล้อม (Environment detection: Colab vs Local macOS / Cloud GPU) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"กำลังทำงานบน Colab: {IN_COLAB}")

# ไลบรารีหลักสำหรับ Eq. 4-5 + การตรวจสอบความถูกต้อง: torch, numpy, scikit-learn
# บน Colab มักจะถูกติดตั้งไว้แล้ว แต่ใส่โค้ดป้องกันไว้เพื่อความปลอดภัย
if IN_COLAB:
    import importlib, subprocess, sys
    for pkg, mod in [("numpy", "numpy"), ("scikit-learn", "sklearn"), ("torch", "torch")]:
        if importlib.util.find_spec(mod) is None:
            print(f"กำลังติดตั้ง {pkg} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
# สำหรับ Local macOS สมมติว่ามี venv ที่ติดตั้ง torch/numpy/scikit-learn เรียบร้อยแล้ว
# หากยังไม่มี ให้รันคำสั่ง: pip install torch numpy scikit-learn

กำลังทำงานบน Colab: False


## 2. การเลือกอุปกรณ์ประมวลผลอัตโนมัติ (Device Auto-select: `cuda` → `mps` → `cpu`)

เลือกลำดับความสำคัญโดยใช้ Cloud GPU (CUDA) ก่อน หากไม่มีจะสลับไปใช้ Apple MPS บน Mac และถ้าไม่มีอีกจะใช้ CPU ซึ่งเป็นการจัดลำดับแบบเดียวกับ `Experiments/Qwen2-VL.py`

In [2]:
import numpy as np
import torch

def select_device() -> str:
    """ฟังก์ชันสำหรับเลือก อุปกรณ์ประมวลผล (Device) ที่เหมาะสมที่สุดตามฮาร์ดแวร์ที่มี"""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = select_device()
print(f"เวอร์ชัน torch : {torch.__version__}")
print(f"อุปกรณ์ที่ใช้  : {DEVICE}")

เวอร์ชัน torch : 2.12.0
อุปกรณ์ที่ใช้  : mps


## 3. นิยามผู้เชี่ยวชาญ (Expert Definitions - Paper Sec. 3)

ผู้เชี่ยวชาญที่มีหน้าที่ต่างกันทั้ง 4 คน (Heterogeneous Experts) จัดเรียงตามลำดับมาตรฐาน (มิติ Output ของ Router จะเรียงตามลำดับนี้):

1. **Reference Extractor** — ดึงภาพปกติที่คล้ายกันที่สุดขึ้นมา (Visual Grounding)
2. **Knowledge Guide** — แทรกความรู้เฉพาะทางของวัตถุนั้นๆ (Domain Knowledge)
3. **Reasoning Expert** — เพิ่มโครงสร้างการคิดแบบเป็นขั้นตอน (Chain-of-Thought Scaffold)
4. **Decision Maker** — ตัวสังเคราะห์ที่ทำงานตลอดเวลาเพื่อบังคับให้ได้คำตอบสุดท้าย (Always-on Synthesiser)

In [3]:
# --- ค่าคงที่ของผู้เชี่ยวชาญ (เรียงตามลำดับมาตรฐาน == ลำดับ Output ของ Router) ---
REFERENCE_EXTRACTOR = "Reference Extractor"
KNOWLEDGE_GUIDE     = "Knowledge Guide"
REASONING_EXPERT    = "Reasoning Expert"
DECISION_MAKER      = "Decision Maker"
EXPERT_NAMES = [REFERENCE_EXTRACTOR, KNOWLEDGE_GUIDE, REASONING_EXPERT, DECISION_MAKER]
N_EXPERTS = len(EXPERT_NAMES)


def activation_vector(active) -> np.ndarray:
    """สร้างเวกเตอร์การเปิดใช้งานแบบทวิภาค (Binary activation vector) y ∈ {0,1}^N ตามรายชื่อผู้เชี่ยวชาญที่เปิดใช้งาน"""
    active = set(active)
    return np.array([1.0 if name in active else 0.0 for name in EXPERT_NAMES],
                    dtype=np.float64)


# --- ค่าเกณฑ์การตัดสินใจแยกตามผู้เชี่ยวชาญ τ_i (Eq. 4 -> การตัดสินใจแบบทวิภาค) ---
def _as_tau_vector(tau) -> np.ndarray:
    """กระจาย (Broadcast) หรือตรวจสอบค่า Threshold ให้อยู่ในรูปเวกเตอร์ขนาด N_EXPERTS

    รองรับทั้งค่าสเกลาร์ (Scalar: ใช้ τ เดียวกันหมด) หรือ List / np.ndarray / torch.Tensor ความยาว N_EXPERTS
    เพื่อช่วยให้สามารถปรับแต่งค่า τ_i ของผู้เชี่ยวชาญแต่ละคนได้อย่างเป็นอิสระในภายหลัง เช่น [0.4, 0.6, 0.5, 0.5]
    """
    if isinstance(tau, torch.Tensor):
        tau = tau.detach().cpu().numpy()
    tau = np.asarray(tau, dtype=np.float64)
    if tau.ndim == 0:                      # สเกลาร์ -> กระจายให้ครบทุกตัว
        tau = np.full(N_EXPERTS, float(tau))
    if tau.shape != (N_EXPERTS,):
        raise ValueError(f"tau ต้องเป็นสเกลาร์ หรือมีมิติเป็น ({N_EXPERTS},) แต่ได้ {tau.shape}")
    return tau


def apply_threshold(probs, tau=0.5) -> np.ndarray:
    """แปลงค่าความน่าจะเป็นแบบ Sigmoid (p_i) -> การตัดสินใจแบบทวิภาค (y_i)

        y_i = 1  ถ้า p_i > tau_i   ไม่เช่นนั้น  0

    `probs` สามารถมีมิติเป็น (N_EXPERTS,) หรือแบบกลุ่ม (n, N_EXPERTS) ก็ได้
    `tau` อาจเป็นสเกลาร์ (ค่าเริ่มต้น 0.5 สำหรับทุกตัว) หรือเวกเตอร์แยกตามผู้เชี่ยวชาญ
    คืนค่าเป็น Array ชนิดจำนวนเต็ม (Integer) ที่มีมิติเดียวกับ `probs`
    """
    if isinstance(probs, torch.Tensor):
        probs = probs.detach().cpu().numpy()
    probs = np.asarray(probs, dtype=np.float64)
    tau_vec = _as_tau_vector(tau)          # (N_EXPERTS,) -> กระจายข้ามแถวทั้งหมด
    return (probs > tau_vec).astype(int)


def experts_from_vector(vec, tau=0.5):
    """คืนค่ารายชื่อผู้เชี่ยวชาญที่มีค่าการเปิดใช้งานเป็น 1 (หลังผ่านการประมวลผลด้วย Threshold)"""
    y = apply_threshold(np.asarray(vec, dtype=np.float64).reshape(-1), tau)
    return [name for name, v in zip(EXPERT_NAMES, y) if v == 1]


# ค่า Threshold เริ่มต้น: τ_i = 0.5 สำหรับผู้เชี่ยวชาญทุกคน (สามารถปรับแยกอิสระภายหลังได้)
DEFAULT_TAU = np.full(N_EXPERTS, 0.5)
print("รายชื่อผู้เชี่ยวชาญ :", EXPERT_NAMES)
print("ค่า Tau เริ่มต้น    :", DEFAULT_TAU.tolist())
print("ตัวอย่าง Tau แบบกำหนดเอง:", _as_tau_vector([0.4, 0.6, 0.5, 0.5]).tolist())

รายชื่อผู้เชี่ยวชาญ : ['Reference Extractor', 'Knowledge Guide', 'Reasoning Expert', 'Decision Maker']
ค่า Tau เริ่มต้น    : [0.5, 0.5, 0.5, 0.5]
ตัวอย่าง Tau แบบกำหนดเอง: [0.4, 0.6, 0.5, 0.5]


## 4. Router Network — Eq. 4  `p = sigmoid(MLP(V_fuse))`

โครงสร้าง MLP ขนาดเล็กทำหน้าที่แปลง Fused Embedding ให้เป็นค่า **Logits** ของผู้เชี่ยวชาญแต่ละคน จากนั้นใช้ฟังก์ชัน **Sigmoid** เพื่อคำนวณค่าความน่าจะเป็นในการเปิดใช้งานที่เป็นอิสระต่อกัน `p ∈ [0,1]^N` (เป็นแบบ Multi-label ไม่ใช่ Softmax Simplex) 

สถาปัตยกรรมเริ่มต้น: `1152 → 512 → 256 → 4`, ใช้ฟังก์ชันเปิดปิด ReLU + Dropout Rate 0.2

- `logits(x)` — ผลลัพธ์ดิบก่อนผ่าน Sigmoid ซึ่งนำไปใช้กับ `BCEWithLogitsLoss` (Eq. 5)
- `predict_proba(x)` — ฟังก์ชันอำนวยความสะดวกสำหรับ NumPy โดยจะรันใน **Eval Mode** (ปิด Dropout) เพื่อให้ผลลัพธ์คงที่แน่นอน (Deterministic) การเรียก `__call__` บน NumPy Array จะถูกส่งมาทำงานที่นี่ (รองรับการใช้งานกับ SHAP)

In [4]:
from typing import List, Optional
from torch import nn


class RouterMLP(nn.Module):
    """Multi-label gating MLP: V_fuse (1152) -> p (N), p = sigmoid(MLP(V_fuse))"""

    def __init__(self, in_dim: int = 1152, n_experts: int = N_EXPERTS,
                 hidden=(512, 256), dropout: float = 0.2) -> None:
        super().__init__()
        self.in_dim = int(in_dim)
        self.n_experts = int(n_experts)
        self.hidden = tuple(hidden)
        self.dropout = float(dropout)

        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_experts))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Eq. 4: ความน่าจะเป็นในการเปิดใช้งาน sigmoid(logits)"""
        return torch.sigmoid(self.net(x))

    def logits(self, x: torch.Tensor) -> torch.Tensor:
        """คืนค่า Logits ดิบก่อนเข้า Sigmoid"""
        return self.net(x)

    # -- ฟังก์ชันอำนวยความสะดวกสำหรับ NumPy (กำหนดให้ผลคงที่: Eval Mode + no_grad) ---------------
    def predict_proba(self, v_fuse: np.ndarray) -> np.ndarray:
        self.eval()
        device = next(self.parameters()).device
        x = torch.as_tensor(np.atleast_2d(v_fuse), dtype=torch.float32, device=device)
        with torch.no_grad():
            return self.forward(x).cpu().numpy()

    def __call__(self, x):  # รักษาให้การรับค่า NumPy Array ทำงานบนเส้นทาง Deterministic
        if isinstance(x, np.ndarray):
            return self.predict_proba(x)
        return super().__call__(x)


# ทดสอบสร้างและรันโมเดลเบื้องต้น (Smoke check): สร้าง Router และรัน Forward Pass 1 ครั้ง
_router = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
_p = _router(np.zeros((1, 1152), dtype=np.float32))
print(f"RouterMLP พร้อมใช้งาน: in_dim=1152 -> รูปทรง p {_p.shape}, p={np.round(_p, 3).tolist()}")

RouterMLP พร้อมใช้งาน: in_dim=1152 -> รูปทรง p (1, 4), p=[[0.4950000047683716, 0.49399998784065247, 0.5, 0.5080000162124634]]


## 5. การฝึกสอนโมเดล (Training) — Eq. 5 (Multi-label BCE)

Router ถูกฝึกสอนด้วย **`BCEWithLogitsLoss`** (Binary Cross Entropy แยกอิสระตามผู้เชี่ยวชาญ) โดยใช้ **`pos_weight`** แยกตามผู้เชี่ยวชาญ (ส่วนกลับของความถี่คลาสที่เป็นบวก / Inverse Positive Frequency) เพื่อแก้ปัญหาคลาสไม่สมดุล (Class Imbalance) ตัวปรับแต่งพารามิเตอร์ (Optimiser): Adam, `lr=1e-3`, `weight_decay=1e-4` 

จากนั้นฟังก์ชัน `select_threshold` จะค้นหาค่า τ **เพียงค่าเดียว** บนชุดข้อมูลประเมินผล (Eval Split) ด้วยวิธี Grid Search เพื่อหาค่าที่ให้ Macro-F1 สูงที่สุด (เป็นจุดเริ่มต้นเพื่อความสะดวก โดยการใช้ `apply_threshold` สามารถนำไปปรับจูน `τ_i` ของแต่ละผู้เชี่ยวชาญได้อย่างอิสระในภายหลัง)

In [5]:
from typing import Tuple


def train_router(X, Y, is_train, hidden=(512, 256), dropout=0.2, epochs=200,
                 lr=1e-3, weight_decay=1e-4, batch_size=64, seed=0,
                 device=None, verbose=True):
    """ฝึกสอน RouterMLP ด้วย BCE (Eq. 5) คืนค่าเป็น (model, loss_history)"""
    torch.manual_seed(seed)
    device = device or DEVICE

    Xtr = torch.tensor(X[is_train], dtype=torch.float32, device=device)
    Ytr = torch.tensor(Y[is_train], dtype=torch.float32, device=device)
    model = RouterMLP(in_dim=X.shape[1], n_experts=Y.shape[1],
                      hidden=hidden, dropout=dropout).to(device)

    pos = Ytr.mean(dim=0).clamp(1e-3, 1 - 1e-3)          # อัตราส่วนคลาสที่เป็นบวกแยกตามผู้เชี่ยวชาญ
    pos_weight = ((1 - pos) / pos).to(device)             # ค่าน้ำหนักตามส่วนกลับของความถี่ (Inverse-frequency weighting)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    n = Xtr.shape[0]
    history: List[float] = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        epoch_loss = 0.0
        for s in range(0, n, batch_size):
            idx = perm[s:s + batch_size]
            opt.zero_grad()
            loss = criterion(model.logits(Xtr[idx]), Ytr[idx])
            loss.backward()
            opt.step()
            epoch_loss += float(loss.detach()) * idx.numel()
        history.append(epoch_loss / max(1, n))
        if verbose and (epoch % 25 == 0 or epoch == epochs - 1):
            print(f"[train] รอบที่ {epoch:3d}  BCE={history[-1]:.4f}")
    return model, history


def macro_f1(y_true, y_pred) -> float:
    """คำนวณค่า Macro-averaged F1 ข้ามผู้เชี่ยวชาญทุกตัว จากการตัดสินใจเชิงทวิภาค (BINARY decisions)"""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    f1s = []
    for j in range(y_true.shape[1]):
        tp = np.sum((y_pred[:, j] == 1) & (y_true[:, j] == 1))
        fp = np.sum((y_pred[:, j] == 1) & (y_true[:, j] == 0))
        fn = np.sum((y_pred[:, j] == 0) & (y_true[:, j] == 1))
        denom = 2 * tp + fp + fn
        f1s.append(1.0 if denom == 0 else (2 * tp) / denom)
    return float(np.mean(f1s))


def select_threshold(model, X, Y, is_train) -> Tuple[float, float]:
    """ค้นหาค่า τ เดี่ยวที่ดีที่สุดด้วย Grid Search บนชุดประเมินผล เพื่อเพิ่มค่า Macro-F1 ให้สูงสุด"""
    eval_mask = ~is_train
    if eval_mask.sum() == 0:
        eval_mask = np.ones(len(is_train), dtype=bool)
    probs = model.predict_proba(X[eval_mask])
    ytrue = Y[eval_mask]
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.linspace(0.1, 0.9, 33):
        f1 = macro_f1(ytrue, apply_threshold(probs, float(tau)))
        if f1 > best_f1:
            best_f1, best_tau = f1, float(tau)
    return best_tau, best_f1


# --- ชุดข้อมูลจำลองแบบกำหนดผลลัพธ์ได้ พร้อมกฎเชิงเส้นที่ทราบค่า (Deterministic synthetic dataset with KNOWN linear rule) ---
# Router ต้องเรียนรู้การเปิดใช้งานผู้เชี่ยวชาญจาก V_fuse ดังนั้นจึงสร้าง Label ด้วยตัวสอนแบบสุ่มเชิงเส้นคงที่
# โดยกำหนดให้ Decision Maker ทำงานตลอดเวลา (Always-on)
def make_synthetic(n=1200, d=1152, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d)).astype(np.float32)
    W = rng.standard_normal((d, N_EXPERTS)).astype(np.float32) / np.sqrt(d)
    b = rng.standard_normal(N_EXPERTS).astype(np.float32) * 0.1
    logits = X @ W + b
    Y = (logits > 0).astype(np.float32)
    Y[:, EXPERT_NAMES.index(DECISION_MAKER)] = 1.0        # Decision Maker เปิดใช้งานตลอดเวลา
    is_train = np.zeros(n, dtype=bool)
    is_train[: int(0.7 * n)] = True                       # แบ่ง 70% เทรน / 30% ประเมินผล
    return X, Y, is_train


print("ฟังก์ชันสำหรับการเทรนพร้อมใช้งาน (train_router, macro_f1, select_threshold, make_synthetic)")

ฟังก์ชันสำหรับการเทรนพร้อมใช้งาน (train_router, macro_f1, select_threshold, make_synthetic)


## 6. *(ส่วนเสริม - Optional)* CLIP Encoder ของจริง — Eq. 1-3

สร้าง `V_fuse` ขนาด 1152 มิติ **ของจริง** จากภาพ + คำถาม โดยใช้ **Frozen CLIP ViT-B/16** พร้อม Projection Layer แบบสุ่มและถูกแช่แข็ง (Deterministic Frozen Projections) ขนาด `512 → 576` สำหรับแต่ละ Modality (ตาม Paper `d = 576`) แล้วเชื่อมต่อกันด้วย `V_fuse = [V_img ; V_text]`

> **ต้องใช้** แพ็กเกจ `clip` (`pip install git+https://github.com/openai/CLIP.git`)
> เซลล์นี้จะ **ข้ามการทำงานอัตโนมัติ** พร้อมแสดงข้อความหากไม่มีแพ็กเกจ `clip` โดยส่วนระบบหลัก Router (Eq. 4-5) และการตรวจสอบความถูกต้องจะไม่ได้รับผลกระทบใดๆ

In [7]:
# เส้นทางประมวลผลแบบ End-to-end เสริม: สร้าง V_fuse ของจริงจาก CLIP (จะข้ามอัตโนมัติถ้าไม่มี clip)
CLIP_NATIVE_DIM = 512   # ขนาด Embedding ของ OpenAI CLIP ViT-B/16
PAPER_DIM       = 576   # มิติ d แต่ละ Modality ใน Paper

class MoXpertEncoder:
    """Frozen CLIP ViT-B/16 + Projections 512->576 คงที่ -> V_fuse (1152)"""

    def __init__(self, clip_model_name="ViT-B/16", target_dim=PAPER_DIM,
                 device=None, proj_seed=1234):
        import clip
        self._clip = clip
        self.device = device or DEVICE
        self.model, self.preprocess = clip.load(clip_model_name, device=self.device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.native_dim = int(self.model.visual.output_dim)
        self.target_dim = int(target_dim)
        g = torch.Generator().manual_seed(proj_seed)
        self.img_proj = self._make_projection(self.native_dim, self.target_dim, g)
        self.text_proj = self._make_projection(self.native_dim, self.target_dim, g)

    def _make_projection(self, in_dim, out_dim, gen):
        if in_dim == out_dim:
            return nn.Identity()
        proj = nn.Linear(in_dim, out_dim, bias=False)
        with torch.no_grad():
            w = torch.empty(out_dim, in_dim)
            w.normal_(0.0, 1.0 / np.sqrt(in_dim), generator=gen)
            proj.weight.copy_(w)
        for p in proj.parameters():
            p.requires_grad_(False)
        return proj.to(self.device)

    def _to_pil(self, image):
        from PIL import Image
        if hasattr(image, "size") and not hasattr(image, "ndim"):
            return image
        if isinstance(image, (str, bytes)):
            return Image.open(image).convert("RGB")
        return image

    def encode_image(self, image) -> np.ndarray:
        x = self.preprocess(self._to_pil(image)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat = self.model.encode_image(x)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            v = self.img_proj(feat.float())
        return v.detach().cpu().numpy().reshape(-1)

    def encode_text(self, text: str) -> np.ndarray:
        tokens = self._clip.tokenize([text], truncate=True).to(self.device)
        with torch.no_grad():
            feat = self.model.encode_text(tokens)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            v = self.text_proj(feat.float())
        return v.detach().cpu().numpy().reshape(-1)

    @staticmethod
    def fuse(v_img, v_text) -> np.ndarray:
        """Eq. 3: V_fuse = [V_img ; V_text]"""
        return np.concatenate([np.asarray(v_img).reshape(-1),
                               np.asarray(v_text).reshape(-1)])

    def encode(self, image, text: str) -> np.ndarray:
        return self.fuse(self.encode_image(image), self.encode_text(text))


try:
    import clip  # noqa: F401
    _HAS_CLIP = True
except ImportError:
    _HAS_CLIP = False

if _HAS_CLIP:
    from PIL import Image
    encoder = MoXpertEncoder(device=DEVICE)
    demo_img = Image.new("RGB", (224, 224), (127, 127, 127))   # ภาพสีเทาจำลอง
    v_fuse = encoder.encode(demo_img, "วัตถุนี้มีจุดบกพร่องหรือไม่?")
    print(f"ขนาดของ V_fuse: {v_fuse.shape} (ควรเป็น (1152,))")
    p = _router(v_fuse[None, :].astype(np.float32))
    print("ค่า p จาก Router บน V_fuse จริง:", np.round(p, 3).tolist())
    print("ผู้เชี่ยวชาญที่เปิดใช้งาน (tau=0.5):", experts_from_vector(p[0], tau=0.5))
else:
    print("[ข้าม] ไม่พบแพ็กเกจ `clip` -> ข้ามเซลล์ CLIP Encoder เสริม")
    print("       สามารถติดตั้งได้ด้วยคำสั่ง: pip install git+https://github.com/openai/CLIP.git")

ขนาดของ V_fuse: (1152,) (ควรเป็น (1152,))
ค่า p จาก Router บน V_fuse จริง: [[0.4950000047683716, 0.49399998784065247, 0.49900001287460327, 0.5080000162124634]]
ผู้เชี่ยวชาญที่เปิดใช้งาน (tau=0.5): ['Decision Maker']


## 7. การตรวจสอบความถูกต้อง (Verification — โมเดลทำงานตรงตามข้อกำหนดใน Paper หรือไม่?)

ชุดทดสอบที่สมบูรณ์ในตัวเอง (สร้างขึ้นใหม่ในเซลล์นี้ โดยไม่มีการอิมพอร์ตจากไฟล์ทดสอบภายนอก) แต่ละรายการจะแสดงผล `PASS`/`FAIL` และสรุปผลรวมในส่วนท้าย

รายการตรวจสอบ:
1. **มิติข้อมูล (Shape)** — ค่าความน่าจะเป็น `p` และผลการตัดสินใจทวิภาค `y` ต้องมีขนาด `(n, 4)`
2. **ขอบเขตข้อมูล (Range)** — ทุกค่าของ `p ∈ [0, 1]`
3. **Multi-label ≠ Softmax** — ผลรวมในแต่ละแถวของ `p` ต้อง **ไม่เท่ากับ** 1
4. **ความแน่นอน (Deterministic in Eval Mode)** — การเรียก `predict_proba` ซ้ำต้องได้ผลลัพธ์ตรงกันเสมอ (ปิด Dropout)
5. **การบันทึก/โหลด `state_dict`** — การบันทึกและโหลดกลับมาต้องได้ผลลัพธ์เท่าเดิม
6. **ค่า BCE ลดลง** — ค่า Loss จากการฝึกสอนลดลงอย่างมีนัยสำคัญ
7. **ค่า F1 > 0.6** — ค่า Macro-F1 ที่คำนวณจาก **การตัดสินใจทวิภาค `y_i`** บนชุดประเมินผลจำลองต้องมากกว่า 0.6

In [8]:
# =========================================================================
# ชุดทดสอบความถูกต้องสมบูรณ์ในตัวเอง (Self-contained verification suite)
# =========================================================================
results = []
def check(name, passed):
    results.append((name, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

torch.manual_seed(0)
np.random.seed(0)

# ---- 1 & 2: ตรวจสอบ Shape + Range ทั้งความน่าจะเป็น p และการตัดสินใจทวิภาค y ------
net = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
Xb = np.random.randn(5, 1152).astype(np.float32)
p = net.predict_proba(Xb)
y = apply_threshold(p, tau=DEFAULT_TAU)                     # ใช้ค่า τ แยกตามผู้เชี่ยวชาญ (ค่าเริ่มต้น 0.5)
check("shape: p มีขนาดเป็น (5, 4)", p.shape == (5, N_EXPERTS))
check("shape: binary decision y มีขนาดเป็น (5, 4)", y.shape == (5, N_EXPERTS))
check("dims: in_dim==2*576==1152 และ n_experts==4",
      net.in_dim == 2 * 576 == 1152 and net.n_experts == 4)
check("range: ทุกค่าของ p อยู่ใน [0, 1]", float(p.min()) >= 0.0 and float(p.max()) <= 1.0)
check("binary: ค่าของ y อยู่ในเซต {0, 1}", set(np.unique(y)).issubset({0, 1}))

# ---- 3: Multi-label (Sigmoid) != Softmax --------------------------------
# Sigmoid อิสระ: ผลรวมของแถวโดยทั่วไปจะไม่เท่ากับ 1 (ขณะที่ Softmax จะบังคับให้เท่ากับ 1)
row_sums = p.sum(axis=1)
check("multi-label != softmax (ผลรวมของแถว != 1)",
      np.any(np.abs(row_sums - 1.0) > 1e-3))

# ---- ตรวจสอบการทำงานของ Threshold แยกตามผู้เชี่ยวชาญ -------------------------
# การเพิ่ม τ ของผู้เชี่ยวชาญตัวใดตัวหนึ่งเข้าใกล้ 1 จะต้องทำให้การตัดสินใจใน คอลัมน์นั้น เป็น 0 ทั้งหมด
tau_custom = np.array([0.5, 0.5, 0.5, 0.999])
y_custom = apply_threshold(p, tau=tau_custom)
check("per-expert tau: เมื่อเพิ่ม tau_3 -> คอลัมน์ที่ 3 กลายเป็น 0 ทั้งหมด",
      bool(np.all(y_custom[:, 3] == 0)))

# ---- 4: ให้ผลคงที่เมื่ออยู่ใน Eval Mode (ปิดการทำงานของ Dropout) -------------------
net_do = RouterMLP(in_dim=1152, n_experts=N_EXPERTS, dropout=0.5).to(DEVICE)
p1 = net_do.predict_proba(Xb)
p2 = net_do.predict_proba(Xb)
check("deterministic in eval (dropout=0.5)", np.allclose(p1, p2, atol=1e-6))

# ---- 5: บันทึกและโหลด state_dict (Round-trip) -------------------------------
import io
buf = io.BytesIO()
torch.save(net.state_dict(), buf)
buf.seek(0)
net_reloaded = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
net_reloaded.load_state_dict(torch.load(buf, map_location=DEVICE))
check("save/load state_dict ให้ผลลัพธ์ตรงกันแม่นยำ",
      np.allclose(net.predict_proba(Xb), net_reloaded.predict_proba(Xb), atol=1e-6))

# ---- 6 & 7: ค่า BCE ลดลงหลังฝึกสอน และค่า F1 > 0.6 จากการตัดสินใจเชิงทวิภาค -
Xs, Ys, is_train = make_synthetic(n=1200, d=1152, seed=0)
model, history = train_router(Xs, Ys, is_train, epochs=120, seed=0, verbose=False)
check("BCE loss ลดลงหลังฝึกสอน (< 0.8 เท่าของค่าเริ่มต้น)",
      history[-1] < history[0] * 0.8)

eval_mask = ~is_train
probs_eval = model.predict_proba(Xs[eval_mask])
y_pred = apply_threshold(probs_eval, tau=DEFAULT_TAU)       # การตัดสินใจแบบ BINARY y_i
f1 = macro_f1(Ys[eval_mask], y_pred)
print(f"       (เทรน BCE {history[0]:.4f} -> {history[-1]:.4f}; ประเมินผล macro-F1 = {f1:.3f})")
check("macro-F1 > 0.6 บนชุดประเมินผลจำลอง (คำนวณจาก binary y_i)", f1 > 0.6)

# ---- สรุปผลการทดสอบ (Summary) --------------------------------------------
n_pass = sum(ok for _, ok in results)
n_total = len(results)
print("\n" + "=" * 56)
overall = "PASS" if n_pass == n_total else "FAIL"
print(f"=== ผลลัพธ์รวม: {overall}  (ผ่าน {n_pass}/{n_total} การทดสอบ) ===")
print("=" * 56)
if overall != "PASS":
    for name, ok in results:
        if not ok:
            print(f"  ล้มเหลว -> {name}")

[PASS] shape: p มีขนาดเป็น (5, 4)
[PASS] shape: binary decision y มีขนาดเป็น (5, 4)
[PASS] dims: in_dim==2*576==1152 และ n_experts==4
[PASS] range: ทุกค่าของ p อยู่ใน [0, 1]
[PASS] binary: ค่าของ y อยู่ในเซต {0, 1}
[PASS] multi-label != softmax (ผลรวมของแถว != 1)
[PASS] per-expert tau: เมื่อเพิ่ม tau_3 -> คอลัมน์ที่ 3 กลายเป็น 0 ทั้งหมด
[PASS] deterministic in eval (dropout=0.5)
[PASS] save/load state_dict ให้ผลลัพธ์ตรงกันแม่นยำ
[PASS] BCE loss ลดลงหลังฝึกสอน (< 0.8 เท่าของค่าเริ่มต้น)
       (เทรน BCE 0.4757 -> 0.0006; ประเมินผล macro-F1 = 0.772)
[PASS] macro-F1 > 0.6 บนชุดประเมินผลจำลอง (คำนวณจาก binary y_i)

=== ผลลัพธ์รวม: PASS  (ผ่าน 11/11 การทดสอบ) ===
